In [1]:
import torch
print(torch.__version__)

2.13.0+cpu


In [2]:
from sdv.single_table import CTGANSynthesizer
print("CTGAN is working!")

CTGAN is working!


1. Install all the required libraries

In [3]:
import pandas as pd
import numpy as np

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

print("Libraries loaded successfully!")

Libraries loaded successfully!


2. Load the preprocessed Cert dataset

In [6]:
data = pd.read_csv(r"C:\Users\Admin\OneDrive - Sol Plaatje University\Documents\cert_preprocessed.csv")

print("Dataset Shape:", data.shape)

data.head()

Dataset Shape: (248301, 17)


,timestamp,user_id,location,action,hour,day,month,weekday,after_hours,employee_name,email,role,business_unit,functional_unit,department,team,supervisor
0,2010-01-02 06:49:00,NGF0157,PC-6056,logon,6,2,1,Saturday,0,Nissim Gil French,Nissim.Gil.French@dtaa.com,FinancialAnalyst,1,4 - Finance,3 - FinancialPlanning,Unknown,Libby Rosalyn Richard
1,2010-01-02 06:50:00,LRR0148,PC-4275,logon,6,2,1,Saturday,0,Libby Rosalyn Richard,Libby.Rosalyn.Richard@dtaa.com,Manager,1,4 - Finance,3 - FinancialPlanning,Unknown,Thomas Vladimir Stokes
2,2010-01-02 06:53:04,LRR0148,PC-4124,logon,6,2,1,Saturday,0,Libby Rosalyn Richard,Libby.Rosalyn.Richard@dtaa.com,Manager,1,4 - Finance,3 - FinancialPlanning,Unknown,Thomas Vladimir Stokes
3,2010-01-02 07:00:00,IRM0931,PC-7188,logon,7,2,1,Saturday,0,Ignatius Reese Morton,Ignatius.Reese.Morton@dtaa.com,ComputerProgrammer,1,2 - ResearchAndEngineering,2 - SoftwareManagement,6 - Software,Caleb Mason Ball
4,2010-01-02 07:00:00,MOH0273,PC-6699,logon,7,2,1,Saturday,0,Macaulay Otto Hopkins,Macaulay.Otto.Hopkins@dtaa.com,HardwareEngineer,1,2 - ResearchAndEngineering,3 - Engineering,4 - SystemsEngineering,Gage Kaden Odonnell


3. Removing the columns that i dont want CTGAN to learn

In [7]:
data = data.drop(columns=[
    "employee_name",
    "email"
], errors="ignore")

print(data.columns)

Index(['timestamp', 'user_id', 'location', 'action', 'hour', 'day', 'month',
       'weekday', 'after_hours', 'role', 'business_unit', 'functional_unit',
       'department', 'team', 'supervisor'],
      dtype='object')


4. Filling in the missing values

In [8]:
for col in data.columns:

    if data[col].dtype == "object":
        data[col] = data[col].fillna("Unknown")

    else:
        data[col] = data[col].fillna(data[col].median())

In [9]:
print(data.isnull().sum())

timestamp          0
user_id            0
location           0
action             0
hour               0
day                0
month              0
weekday            0
after_hours        0
role               0
business_unit      0
functional_unit    0
department         0
team               0
supervisor         0
dtype: int64


5. Convert timestamp to date time

In [10]:
data["timestamp"] = pd.to_datetime(data["timestamp"])

6. Detecting metadata

In [11]:
metadata = SingleTableMetadata()

metadata.detect_from_dataframe(data)

metadata

{
    "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1",
    "columns": {
        "timestamp": {
            "sdtype": "datetime"
        },
        "user_id": {
            "sdtype": "id"
        },
        "location": {
            "sdtype": "categorical"
        },
        "action": {
            "sdtype": "categorical"
        },
        "hour": {
            "sdtype": "numerical"
        },
        "day": {
            "sdtype": "numerical"
        },
        "month": {
            "sdtype": "categorical"
        },
        "weekday": {
            "sdtype": "categorical"
        },
        "after_hours": {
            "sdtype": "categorical"
        },
        "role": {
            "sdtype": "categorical"
        },
        "business_unit": {
            "sdtype": "categorical"
        },
        "functional_unit": {
            "sdtype": "categorical"
        },
        "department": {
            "sdtype": "categorical"
        },
        "team": {
            "sdtype": "categorical"


7. Building CTGAN model

In [21]:
ctgan = CTGANSynthesizer(

    metadata=metadata,

    epochs=300,

    verbose=True

)

8. Train the Model

In [22]:
ctgan.fit(data)

print("Training completed!")

PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
timestamp              11
location               1002
action                 5
hour                   11
day                    11
month                  2
weekday                7
after_hours            2
role                   42
business_unit          1
functional_unit        7
department             23
team                   39
supervisor             68

We recommend preprocessing discrete columns that can have many values, using 'update_transformers'. Or you may drop columns that are not necessary to model. (Exit this script using ctrl-C)


MemoryError: Unable to allocate 1.85 GiB for an array with shape (248301, 1002) and data type int64

9. Generate the synthetic dataset

In [ ]:
synthetic_ctgan = ctgan.sample(num_rows=len(data))

synthetic_ctgan.head()